# Modeling Criteria EDA

Which findings from the normative-v2 exploration actually became engine thresholds, and were they justified? All thresholds below are imported from `config.MODELING_PARAMS_V2` / `Modeling.dispersion_trend` -- nothing here is re-derived independently of the engine.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import statsmodels.api as sm
from scipy.sparse import issparse
from scipy.stats import norm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

_ROOT = Path.cwd().parent
for p in (_ROOT, _ROOT / "Modeling"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import config
from dispersion_trend import build_trend, load_trend
from model_engine_v2 import _poisson_rqr, _nb_rqr
from viz_style import apply_style

apply_style()

MP2 = config.MODELING_PARAMS_V2
OUT_DIR = Path.cwd() / "Analysis_Results"
FIG_DIR = OUT_DIR / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Engine thresholds in use (config.MODELING_PARAMS_V2):")
for k in ["nz_a_max", "nz_bc_split", "trend_min_nz", "beta_explode_thr"]:
    print(f"  {k} = {MP2[k]}")


In [ ]:
def load_hc():
    adata = sc.read_h5ad(config.H5AD_PATH)
    m = ((adata.obs["QC_Passed"] == True) & (adata.obs["Phenotype_Processed"].notna()) &
         (adata.obs["Phenotype_Processed"] != "Unknown") &
         (adata.obs["broad_protocol_category"] != "Exome-based (EB)"))
    a = adata[m]
    is_hc = (a.obs["Phenotype_Processed"].astype(str) == "Healthy Control").values
    is_pc = (a.var["GeneType"] == "protein_coding").values
    X = a.obs[config.BIAS_COLUMNS].values.astype(np.float64)[is_hc]
    Xs = StandardScaler().fit_transform(X)
    Y = a.X.toarray() if issparse(a.X) else np.asarray(a.X)
    Y = np.round(Y[is_hc][:, is_pc]).astype(np.float64)
    names = np.array(a.var_names[is_pc].tolist())
    strata = a.obs[MP2["stratify_col"]].astype(object).fillna("NA").astype(str).values[is_hc]
    return Xs, Y, names, strata


def cameron_trivedi(y, mu):
    g = (y - mu) ** 2 - y
    x = mu ** 2
    denom = float((x * x).sum())
    if denom <= 0:
        return np.nan, np.nan
    alpha = float((g * x).sum() / denom)
    resid = g - alpha * x
    dof = len(y) - 1
    if dof <= 0:
        return alpha, np.nan
    se = np.sqrt((resid ** 2).sum() / dof / denom)
    if se <= 0 or not np.isfinite(se):
        return alpha, np.nan
    return alpha, float(norm.sf(alpha / se))


def w1_normal(z):
    v = z[np.isfinite(z)]
    n = len(v)
    if n < 8:
        return np.nan
    ref = norm.ppf(np.linspace(1 / (2 * n), 1 - 1 / (2 * n), n))
    return float(np.mean(np.abs(np.sort(v) - ref)))


In [ ]:
Xs, Y, names, strata = load_hc()
name2idx = {g: i for i, g in enumerate(names)}
print(f"HC={Xs.shape[0]}  protein-coding genes={len(names)}")


## 1. Gating landscape

Route sizes at the engine's actual `nz_a_max` / `nz_bc_split`.

In [ ]:
nz = (Y > 0).sum(0).astype(int)
route = np.where(nz < MP2["nz_a_max"], "A",
                np.where(nz < MP2["nz_bc_split"], "B", "C"))
gating_df = pd.DataFrame({"gene": names, "nz": nz, "route": route})
gating_df.to_csv(OUT_DIR / "gating_landscape.csv", index=False)

route_counts = gating_df["route"].value_counts()
print(route_counts)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nz, bins=80, color="#4477aa")
ax.axvline(MP2["nz_a_max"], color="k", ls="--", lw=1, label=f"nz_a_max={MP2['nz_a_max']}")
ax.axvline(MP2["nz_bc_split"], color="#d95f02", ls="--", lw=1, label=f"nz_bc_split={MP2['nz_bc_split']}")
ax.set(xlabel="nonzero HC samples (NZ)", ylabel="genes",
      title=f"Engine gating: A={route_counts.get('A',0)} B={route_counts.get('B',0)} C={route_counts.get('C',0)}")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "gating_landscape.png", dpi=150)
plt.show()


## 2. Route B overdispersion

Cameron-Trivedi NB2 test justifying why Route B cannot be plain Poisson -- the premise for borrowing dispersion from the trend instead of assuming Poisson.

In [ ]:
b_genes = gating_df.loc[gating_df["route"] == "B", "gene"].values
Xc = sm.add_constant(Xs)
rows = []
for g in b_genes:
    y = Y[:, name2idx[g]]
    try:
        res = sm.GLM(y, Xc, family=sm.families.Poisson()).fit()
    except Exception:
        continue
    mu = np.clip(res.mu, 1e-8, None)
    alpha, p = cameron_trivedi(y, mu)
    rows.append({"gene": g, "ct_alpha": alpha, "ct_p": p,
                "pearson_disp": float(res.pearson_chi2 / res.df_resid)})

overdisp_df = pd.DataFrame(rows)
overdisp_df["overdispersed"] = overdisp_df["ct_p"] < 0.05
overdisp_df.to_csv(OUT_DIR / "routeB_overdispersion.csv", index=False)

print(f"Route B genes tested: {len(overdisp_df)}")
print(f"Significantly overdispersed (Cameron-Trivedi p<0.05): {overdisp_df['overdispersed'].mean():.1%}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(overdisp_df["ct_alpha"].clip(-1, 10), bins=60, color="#d95f02")
ax.axvline(0, color="k", ls="--", lw=1)
ax.set(xlabel="NB2 alpha (Cameron-Trivedi, clipped)", ylabel="Route B genes",
      title=f"Route B overdispersion: {overdisp_df['overdispersed'].mean():.1%} significant (p<0.05)\n"
            f"Poisson assumption for Route B is REJECTED by this test")
fig.tight_layout(); fig.savefig(FIG_DIR / "routeB_overdispersion.png", dpi=150)
plt.show()


## 3. Dispersion trend shape: single log-log line (rejected) vs lowess (used by the engine)

`build_trend`/`load_trend` here are the exact functions `model_engine_v2.py` calls -- this section only re-derives the single-line alternative for comparison, it does not change what the engine uses.

In [ ]:
trend = build_trend(Y, min_nz=MP2["trend_min_nz"])  # same call the engine makes
mean_c = Y.mean(0)
var_c = Y.var(0)
with np.errstate(divide="ignore", invalid="ignore"):
    sigma_mom = np.where(mean_c > 0, (var_c - mean_c) / mean_c ** 2, np.nan)
sigma_mom = np.clip(sigma_mom, 0, None)
reliable = (nz >= MP2["trend_min_nz"]) & (mean_c > 0) & np.isfinite(sigma_mom)
mu, sig = mean_c[reliable], sigma_mom[reliable]

pos = sig > 1e-3
lx, ly = np.log(mu[pos]), np.log(sig[pos])
line = sm.OLS(ly, sm.add_constant(lx)).fit()
r2 = float(line.rsquared)

logmu = np.asarray(trend["lowess_logmu"])
logsig = np.asarray(trend["lowess_logsigma"])

trend_comparison = pd.DataFrame({
    "criterion": ["single log-log line (rejected)", "lowess on log-log MoM (used by engine)"],
    "r2_or_note": [f"R2={r2:.3f}, systematically misses both plateaus", "used via dispersion_trend.build_trend"],
})
trend_comparison.to_csv(OUT_DIR / "dispersion_trend_fit_comparison.csv", index=False)
print(trend_comparison.to_string(index=False))

grid = np.geomspace(mu.min(), mu.max(), 200)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mu, sig, s=3, alpha=0.08, color="#bbbbbb", label="per-gene MoM sigma")
ax.plot(grid, np.exp(line.params[0] + line.params[1] * np.log(grid)), color="#7570b3",
       lw=2, label=f"single log-log line (R2={r2:.2f})")
ax.plot(np.exp(logmu), np.exp(logsig), color="#d95f02", lw=2.5,
       label="lowess (engine's dispersion_trend.load_trend)")
ax.set(xscale="log", yscale="log", xlabel="mu (raw mean)", ylabel="sigma (NB2 dispersion)",
      title=f"trend_min_nz={MP2['trend_min_nz']}: line underfits both plateaus, lowess used instead")
ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR / "dispersion_trend_fit_comparison.png", dpi=150)
plt.show()


## 4. NZ_A_MAX identifiability basis (RQR-independent)

Cross-fold coefficient/prediction stability vs NZ -- the basis for the Route A/B threshold. Deliberately does NOT use RQR calibration (section 5 shows why that would be misleading at low NZ).

In [ ]:
alpha_fn = load_trend()  # same trend the engine scores Route B with

NZ_LO, NZ_HI = 5, 90
band = np.where((nz >= NZ_LO) & (nz < NZ_HI))[0]
folds = list(StratifiedKFold(MP2["n_splits"], shuffle=True, random_state=42)
            .split(np.zeros(Xs.shape[0]), strata))

rows = []
for j in band:
    y = Y[:, j]
    mu_folds, beta_max = [], []
    for tr, te in folds:
        alpha_g = alpha_fn(float(y[tr].mean()))
        try:
            fit = sm.GLM(y[tr], Xc[tr], family=sm.families.NegativeBinomial(alpha=alpha_g)).fit()
            mu_folds.append(np.clip(fit.predict(Xc), 1e-8, 1e12))
            beta_max.append(float(np.abs(fit.params[1:]).max()))
        except Exception:
            pass
    if len(mu_folds) < 2:
        continue
    M = np.vstack(mu_folds)
    with np.errstate(divide="ignore", invalid="ignore"):
        cv = M.std(0) / M.mean(0)
    rows.append({"gene": names[j], "nz": int(nz[j]),
                "mu_cv": float(np.nanmedian(cv[np.isfinite(cv)])),
                "beta_max": float(np.nanmax(beta_max)),
                "beta_explode": float(np.nanmax(beta_max)) > MP2["beta_explode_thr"]})

ident_df = pd.DataFrame(rows)
ident_df.to_csv(OUT_DIR / "nz_identifiability.csv", index=False)

edges = sorted(set(e for e in [NZ_LO, 15, 20, 25, MP2["nz_a_max"], 40, 50, 60, MP2["nz_bc_split"], NZ_HI]
                   if NZ_LO <= e <= NZ_HI))
lab = [f"{edges[i]}-{edges[i+1]}" for i in range(len(edges) - 1)]
ident_df["nz_bin"] = pd.cut(ident_df["nz"], bins=edges, labels=lab, right=False, include_lowest=True)
ident_agg = ident_df.groupby("nz_bin", observed=True).agg(
    n=("gene", "size"), median_mu_cv=("mu_cv", "median"),
    frac_beta_explode=("beta_explode", "mean")).reset_index()
print(ident_agg.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = range(len(ident_agg))
axes[0].plot(x, ident_agg["median_mu_cv"], "-o", color="#d95f02")
axes[0].set(ylabel="median cross-fold mu CV", title="Coefficient/prediction instability vs NZ")
axes[1].plot(x, ident_agg["frac_beta_explode"], "-o", color="#7570b3")
axes[1].set(ylabel=f"fraction |beta|>{MP2['beta_explode_thr']}", title="Non-identifiable fraction vs NZ")
for ax in axes:
    ax.set_xticks(list(x)); ax.set_xticklabels(ident_agg["nz_bin"], rotation=60, ha="right")
fig.suptitle(f"nz_a_max={MP2['nz_a_max']}: basis for the Route A/B threshold")
fig.tight_layout(); fig.savefig(FIG_DIR / "nz_identifiability.png", dpi=150)
plt.show()


## 5. INVALID CRITERION: RQR W1 at low NZ

This section exists to document what NOT to use. At low NZ, near-all-zero genes manufacture apparent N(0,1) regardless of whether the underlying model is correct -- so W1 calibration alone cannot distinguish a mis-specified Poisson model from the correct trend-dispersion NB. This is why section 4's coefficient-stability diagnostic, not this one, is the actual criterion the engine's `nz_a_max` is based on.

In [ ]:
rows = []
for j in band:
    y = Y[:, j]
    zp, zn = np.full(len(y), np.nan), np.full(len(y), np.nan)
    for fi, (tr, te) in enumerate(folds):
        alpha_g = alpha_fn(float(y[tr].mean()))
        try:
            pois = sm.GLM(y[tr], Xc[tr], family=sm.families.Poisson()).fit()
            zp[te] = _poisson_rqr(y[te], np.clip(pois.predict(Xc[te]), 1e-8, 1e8), 42 + fi)
        except Exception:
            pass
        try:
            nbf = sm.GLM(y[tr], Xc[tr], family=sm.families.NegativeBinomial(alpha=alpha_g)).fit()
            zn[te] = _nb_rqr(y[te], np.clip(nbf.predict(Xc[te]), 1e-8, 1e8), alpha_g, 42 + fi)
        except Exception:
            pass
    w1p, w1n = w1_normal(zp), w1_normal(zn)
    rows.append({"gene": names[j], "nz": int(nz[j]), "w1_poisson": w1p, "w1_nb_trend": w1n, "gap": w1p - w1n})

rqr_df = pd.DataFrame(rows)
rqr_df.to_csv(OUT_DIR / "rqr_low_nz_bias.csv", index=False)

rqr_df["nz_bin"] = pd.cut(rqr_df["nz"], bins=edges, labels=lab, right=False, include_lowest=True)
rqr_agg = rqr_df.groupby("nz_bin", observed=True).agg(
    n=("gene", "size"), w1_poisson=("w1_poisson", "median"),
    w1_nb_trend=("w1_nb_trend", "median"), gap=("gap", "median")).reset_index()
print(rqr_agg.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(rqr_agg))
ax.plot(x, rqr_agg["w1_poisson"], "-o", color="#7570b3", label="mis-specified Poisson")
ax.plot(x, rqr_agg["w1_nb_trend"], "-o", color="#d95f02", label="trend-dispersion NB")
ax.set_xticks(list(x)); ax.set_xticklabels(rqr_agg["nz_bin"], rotation=60, ha="right")
ax.set(ylabel="median W1 to N(0,1)",
      title="At low NZ, W1 cannot distinguish a wrong model from a correct one\n"
            "(gap widens only once NZ is large enough to actually test the model)")
ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR / "rqr_low_nz_bias.png", dpi=150)
plt.show()
